# Programação Paralela e Distribuída
Terceira Atividade Avaliativa - 23/11/2026 - Etapa 2 - **Modelo B** **(vale 3.0)

Nome: Eduardo Santiago Rosalin

RA: 802539


## Orientações

Atividade em dupla. É vedada a troca de mensagens de qualquer natureza entre grupos. Duração: até 2 aulas.

Use este notebook só se a sua dupla fez o **Modelo B** na Etapa 1. Com base nas respostas da Etapa 1, paralelize o problema com MPI. Serão observados:

* Uso adequado de MPI (resultado correto com qualquer número de processos, distribuição dos dados, comunicação que não trava);
* Organização e clareza do código, incluindo comentários;
* Coerência entre a proposta escrita na Etapa 1 e a implementação; e
* Capacidade de explicar o funcionamento do código desenvolvido (se solicitado).

Regras:

1. **Não altere a geração dos dados, os `#define` nem a fórmula do cálculo** (`novo[i] = ...`). As três primeiras linhas da saída devem ser **idênticas** às da saída esperada, com 1, 2, 3 e 4 processos.
2. Rodar e obter o resultado certo **não prova** que o código está correto: um envio pode funcionar só porque a mensagem é pequena. Justifique, nos comentários, por que a sua comunicação não trava.
3. O Colab tem só 2 núcleos. Com 3 e 4 processos, os processos se revezam nos núcleos: o resultado vale, mas o tempo não mede o ganho do paralelismo.
4. Execute todas as células e **entregue o notebook com as saídas**. Preencha a tabela de tempos com os valores que aparecem nas suas saídas.
5. Se a implementação mostrar que alguma resposta da Etapa 1 estava errada, explique na célula de análise. Isso conta a favor.
6. Caso utilize IA, cole os prompts **completos** na última seção (links não são aceitos).

Entrega: Arquivo → Fazer download → Baixar o .ipynb, com o nome `nomedosalunos.ipynb`. Apenas um da dupla envia.

## Problema – Suavização de um sinal

Um sinal é representado por `N` amostras, `s[2]` a `s[N+1]`, com valores iniciais sorteados entre −100 e 99. Antes do sinal há duas posições fixas em 50 (`s[0]` e `s[1]`), e depois dele duas posições fixas em −50 (`s[N+2]` e `s[N+3]`). A cada iteração, o novo valor de cada amostra é a média dela com as duas anteriores e as duas seguintes. O programa conta quantas amostras variaram pelo menos `TOL` na iteração e para quando nenhuma variou.

Os dados são gerados no rank 0, como se fossem lidos de um arquivo: o sinal deve ser distribuído a partir dele e recolhido nele no fim.

Saída esperada (o tempo varia):
```
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
```

### Versão sequencial (não altere)

In [5]:
%%writefile sinal_seq.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N       1200000        // amostras do sinal (divisivel por 1, 2, 3 e 4)
#define TOL     0.01           // uma amostra "ainda muda" se variar pelo menos isto numa iteracao
#define MAX_IT  100000         // limite de seguranca
#define B_ESQ   50.0           // valor fixo das duas posicoes antes do sinal
#define B_DIR   -50.0          // valor fixo das duas posicoes depois do sinal

int main(int argc, char *argv[]) {
    int rank, nprocs;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &nprocs);

    // Versao atual: so o rank 0 trabalha.
    if (rank == 0) {
        // Amostras do sinal: s[2..N+1]. s[0], s[1], s[N+2] e s[N+3] sao as bordas, que nao mudam.
        double *s    = malloc((N + 4) * sizeof(double));
        double *novo = malloc((N + 4) * sizeof(double));
        srand(7);                           // os dados sao gerados no rank 0
        for (int i = 2; i <= N + 1; i++)
            s[i] = rand() % 200 - 100;      // amostra inicial, entre -100 e 99
        s[0] = s[1] = novo[0] = novo[1] = B_ESQ;
        s[N + 2] = s[N + 3] = novo[N + 2] = novo[N + 3] = B_DIR;

        double t0 = MPI_Wtime();
        int it = 0;
        int mudaram;                        // amostras que variaram pelo menos TOL nesta iteracao
        do {
            mudaram = 0;
            for (int i = 2; i <= N + 1; i++) {
                // novo valor = media da amostra com as duas anteriores e as duas seguintes
                novo[i] = (s[i - 2] + s[i - 1] + s[i] + s[i + 1] + s[i + 2]) / 5.0;
                if (fabs(novo[i] - s[i]) >= TOL)
                    mudaram++;
            }
            double *tmp = s; s = novo; novo = tmp;   // o novo sinal passa a ser o atual
            it++;
        } while (mudaram > 0 && it < MAX_IT);
        double t1 = MPI_Wtime();

        double soma = 0.0;
        for (int i = 2; i <= N + 1; i++)
            soma += s[i];
        printf("Iteracoes = %d\n", it);
        printf("Soma      = %.6f\n", soma);
        printf("s[2] = %.6f   s[N/2] = %.6f   s[N+1] = %.6f\n", s[2], s[N / 2], s[N + 1]);
        printf("Tempo = %.2f s\n", t1 - t0);
        free(s); free(novo);
    }
    MPI_Finalize();
    return 0;
}

Writing sinal_seq.c


In [6]:
!mpicc -O2 -Wall -o sinal_seq sinal_seq.c -lm && mpirun --allow-run-as-root -n 1 ./sinal_seq

Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 5.04 s


In [10]:
!lscpu | grep -E "Model name|CPU\(s\)|Thread|Core|Socket"

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1


### Versão paralela

Lembretes para a troca de fronteiras (B1 da Etapa 1):

* Cada processo guarda o seu bloco em `s[2..n+1]` e reserva células extras dos dois lados (`s[0]`, `s[1]` e `s[n+2]`, `s[n+3]`) para os valores que vêm dos vizinhos. Essas células são atualizadas **a cada iteração**, antes do cálculo.
* No primeiro e no último processo, as células extras do lado de fora do sinal são as bordas fixas (`B_ESQ` ou `B_DIR`).
* `MPI_Sendrecv` envia e recebe na mesma chamada. Com `MPI_PROC_NULL` como parceiro, o lado correspondente da chamada não faz nada: útil para o primeiro e o último processo.
* Fazer `MPI_Send` em todos os processos e só depois `MPI_Recv` pode travar: o envio padrão pode esperar o recebimento do outro lado (Exercício 4 da lista).

In [3]:
%%writefile sinal_par.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#ifndef N
#define N       1200000
#endif
#ifndef TOL
#define TOL     0.01
#endif
#ifndef MAX_IT
#define MAX_IT  100000
#endif
#define B_ESQ   50.0
#define B_DIR  -50.0

int main(int argc, char *argv[]) {
    int rank, nprocs;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &nprocs);


    int n = N / nprocs;
    double *s = malloc((n + 4) * sizeof(double));
    double *novo = malloc((n + 4) * sizeof(double));
    double *dados = NULL;

    if (s == NULL || novo == NULL) {
        fprintf(stderr, "Rank %d: erro de alocacao de memoria.\n", rank);
        // Aborta todos os processos se um deles nao conseguiu alocar memoria.
        MPI_Abort(MPI_COMM_WORLD, 1);
        return 1;
    }

    if (rank == 0) {
        dados = malloc(N * sizeof(double));
        if (dados == NULL) {
            fprintf(stderr, "Erro de alocacao do sinal global.\n");
            MPI_Abort(MPI_COMM_WORLD, 1);
            return 1;
        }
        srand(7);
        for (int i = 0; i < N; i++) {
            dados[i] = rand() % 200 - 100;
        }
    }

    // Distribui N amostras geradas pelo rank 0 em blocos de n amostras.
    MPI_Scatter(dados, n, MPI_DOUBLE,
                &s[2], n, MPI_DOUBLE,
                0, MPI_COMM_WORLD);

    int esquerda;
    int direita;
    if (rank == 0) {
        esquerda = MPI_PROC_NULL;
    } else {
        esquerda = rank - 1;
    }
    if (rank == nprocs - 1) {
        direita = MPI_PROC_NULL;
    } else {
        direita = rank + 1;
    }

    // Todos devem estar na mesma linha antes de começar o while loop
    MPI_Barrier(MPI_COMM_WORLD);
    double t0 = MPI_Wtime();
    int it = 0;
    int mudaram_global;

    do {
        if (rank == 0) {
            s[0] = B_ESQ;
            s[1] = B_ESQ;
        }
        if (rank == nprocs - 1) {
            s[n + 2] = B_DIR;
            s[n + 3] = B_DIR;
        }

        // Envia as duas primeiras amostras para a esquerda; ao mesmo tempo,
        // recebe as duas primeiras amostras do vizinho da direita em seu halo direito.
        // MPI_PROC_NULL faz a operacao correspondente ser ignorada, sem travar.
        MPI_Sendrecv(&s[2], 2, MPI_DOUBLE, esquerda, 0,
                     &s[n + 2], 2, MPI_DOUBLE, direita, 0,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        // Envia as duas ultimas amostras para a direita; ao mesmo tempo,
        // recebe as duas ultimas amostras do vizinho da esquerda em seu halo esquerdo.
        // A troca casada de envio/recebimento evita deadlock.
        MPI_Sendrecv(&s[n], 2, MPI_DOUBLE, direita, 1,
                     &s[0], 2, MPI_DOUBLE, esquerda, 1,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        int mudaram_local = 0;
        for (int i = 2; i <= n + 1; i++) {
            novo[i] = (s[i - 2] + s[i - 1] + s[i] + s[i + 1] + s[i + 2]) / 5.0;
            if (fabs(novo[i] - s[i]) >= TOL) {
                mudaram_local++;
            }
        }

        double *tmp = s;
        s = novo;
        novo = tmp;

        // O divulga o resultado o resultado para TODOS.
        // Assim, nenhum processo para enquanto ainda ha mudancas em outro bloco.
        MPI_Allreduce(&mudaram_local, &mudaram_global, 1,
                      MPI_INT, MPI_SUM, MPI_COMM_WORLD);
        it++;
    } while (mudaram_global > 0 && it < MAX_IT);

    double tempo_local = MPI_Wtime() - t0;
    double tempo_max = 0.0;

    // Coleta o maior tempo entre processos: a execucao paralela so termina
    // quando o ultimo processo conclui seu trabalho.
    MPI_Reduce(&tempo_local, &tempo_max, 1,
               MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);

    // Reunir os dados de todos os processos em um único processo.
    // preservando a ordem original para somar como na versao sequencial.
    MPI_Gather(&s[2], n, MPI_DOUBLE,
               dados, n, MPI_DOUBLE,
               0, MPI_COMM_WORLD);

    if (rank == 0) {
        double soma = 0.0;
        for (int i = 0; i < N; i++) {
            soma += dados[i];
        }
        printf("Iteracoes = %d\n", it);
        printf("Soma      = %.6f\n", soma);

        printf("s[2] = %.6f   s[N/2] = %.6f   s[N+1] = %.6f\n",
               dados[0], dados[N / 2 - 2], dados[N - 1]);
        printf("Tempo = %.2f s\n", tempo_max);
    }

    free(dados);
    free(s);
    free(novo);

    MPI_Finalize();
    return 0;
}


Writing sinal_par.c


In [4]:
!mpicc -O2 -Wall -o sinal_par sinal_par.c -lm
!for p in 1 2 3 4; do echo "== $p processo(s)"; mpirun --allow-run-as-root -n $p -host localhost:4 ./sinal_par; done

== 1 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 5.35 s
== 2 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 7.15 s
== 3 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 9.81 s
== 4 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 10.00 s


In [9]:
# Repetição: o resultado precisa ser o mesmo em todas as execuções
!for r in 1 2 3; do mpirun --allow-run-as-root -n 4 -host localhost:4 ./sinal_par | sed -n 1,2p; done

Iteracoes = 1279
Soma      = -596965.773600
Iteracoes = 1279
Soma      = -596965.773600
Iteracoes = 1279
Soma      = -596965.773600


### Tempos e análise

Preencha com os tempos das suas saídas.

| Versão     | 1 processo | 2 processos | 3 processos | 4 processos |
| ---------- | ---------- | ----------- | ----------- | ----------- |
| Sequencial | 5,04 s     | –           | –           | –           |
| Paralela   | 5,35 s     | 7,15 s      | 9,81 s      | 10,00 s     |


1. O resultado foi idêntico ao esperado com todos os números de processos e em todas as execuções? Por que a troca de fronteiras do seu código não trava, qualquer que seja o tamanho das mensagens?


2. Compare os tempos com 1 e 2 processos com o sequencial. Que ganho vocês esperavam com 2 processos, sabendo que cada um troca só quatro `double` com os vizinhos por iteração? O medido confirmou? Com 3 e 4 processos no Colab, o tempo piora muito. Por quê?



3. O que mudou em relação ao planejamento da Etapa 1, se algo mudou?


*Respostas:*

1. Sim, o resultado foi idêntico ao esperado com todos os números de processos e em todas as execuções. Nao ocorre deadlock porque o MPI_Sendrecv que combina envio e recebimento em uma única chamada.
Além disso, o MPI_PROC_NULL para representar os vizinhos inexistentes nos processos das extremidades.

2. A versão sequencial executou em 5,04 segundos, enquanto a paralela com um processo levou 5,35 segudos. Com dois processos, esperávamos uma redução significativa do tempo, potencialmente em 50%. A comunicação entre vizinhos envolve somente duas amostras por fronteira em cada direção, um volume pequeno em comparação com o trabalho computacional.

A inspeção do ambiente com lscpu indicou recursos limitados de CPU, com apenas um núcleo por socket e duas threads por núcleo. Essa configuração ajuda a explicar a perda de desempenho, embora também seja necessário considerar a quantidade de sockets e eventuais cotas de CPU da máquina virtual.

Como os processos MPI competem pelos recursos disponíveis, a execução pode sofrer com contenção de CPU e escalonamento pelo sistema operacional. Além disso, cada uma das 1.279 iterações exige duas chamadas MPI_Sendrecv e uma MPI_Allreduce, acrescentando custos de comunicação e sincronização. Dependendo da implementação MPI, a espera por essas operações também pode consumir tempo de CPU.


3. O sinal foi distribuído estaticamente em blocos contíguos de tamanho n = N / nprocs, com duas células extras em cada extremidade para armazenar os valores recebidos dos processos vizinhos.

A troca de fronteiras foi implementada com MPI_Sendrecv, enquanto a convergência global foi verificada com MPI_Allreduce. Também mantivemos a troca de ponteiros entre s e novo a cada iteração, evitando cópias desnecessárias dos vetores.

Por fim, utilizamos MPI_Gather para reunir o sinal no processo 0 e calcular a soma na mesma ordem da versão sequencial.

A implementação apresentou os resultados numéricos esperados, embora os testes tenham demonstrado que a paralelização não trouxe ganhos de desempenho nas condições de execução do Google Colab.

In [11]:
!lscpu | grep -E "Model name|CPU\(s\)|Thread|Core|Socket"

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1


## Uso de IA

Se usaram IA, colem aqui os prompts **completos** e as respostas relevantes (links não são aceitos). Se não usaram, escrevam "Não utilizamos IA".

- Preencha em markdown:

```
### Tempos e análise

Preencha com os tempos das suas saídas.

| Versão | 1 processo | 2 processos | 3 processos | 4 processos |
| --- | --- | --- | --- | --- |
| Sequencial |  | – | – | – |
| Paralela |  |  |  |  | Tempos e análise
Versão	1 processo	2 processos	3 processos	4 processos
Sequencial	5,04 s	–	–	–
Paralela	5,35 s	7,15 s	9,81 s	10,00 s

```



- Analisar o speedup e eficiência comparando com a versão sequencial do meu codigo, porque a versao paralela está mais lenta:


```
 %%writefile sinal_seq.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N       1200000        // amostras do sinal (divisivel por 1, 2, 3 e 4)
#define TOL     0.01           // uma amostra "ainda muda" se variar pelo menos isto numa iteracao
#define MAX_IT  100000         // limite de seguranca
#define B_ESQ   50.0           // valor fixo das duas posicoes antes do sinal
#define B_DIR   -50.0          // valor fixo das duas posicoes depois do sinal

int main(int argc, char *argv[]) {
    int rank, nprocs;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &nprocs);

    // Versao atual: so o rank 0 trabalha.
    if (rank == 0) {
        // Amostras do sinal: s[2..N+1]. s[0], s[1], s[N+2] e s[N+3] sao as bordas, que nao mudam.
        double *s    = malloc((N + 4) * sizeof(double));
        double *novo = malloc((N + 4) * sizeof(double));
        srand(7);                           // os dados sao gerados no rank 0
        for (int i = 2; i <= N + 1; i++)
            s[i] = rand() % 200 - 100;      // amostra inicial, entre -100 e 99
        s[0] = s[1] = novo[0] = novo[1] = B_ESQ;
        s[N + 2] = s[N + 3] = novo[N + 2] = novo[N + 3] = B_DIR;

        double t0 = MPI_Wtime();
        int it = 0;
        int mudaram;                        // amostras que variaram pelo menos TOL nesta iteracao
        do {
            mudaram = 0;
            for (int i = 2; i <= N + 1; i++) {
                // novo valor = media da amostra com as duas anteriores e as duas seguintes
                novo[i] = (s[i - 2] + s[i - 1] + s[i] + s[i + 1] + s[i + 2]) / 5.0;
                if (fabs(novo[i] - s[i]) >= TOL)
                    mudaram++;
            }
            double *tmp = s; s = novo; novo = tmp;   // o novo sinal passa a ser o atual
            it++;
        } while (mudaram > 0 && it < MAX_IT);
        double t1 = MPI_Wtime();

        double soma = 0.0;
        for (int i = 2; i <= N + 1; i++)
            soma += s[i];
        printf("Iteracoes = %d\n", it);
        printf("Soma      = %.6f\n", soma);
        printf("s[2] = %.6f   s[N/2] = %.6f   s[N+1] = %.6f\n", s[2], s[N / 2], s[N + 1]);
        printf("Tempo = %.2f s\n", t1 - t0);
        free(s); free(novo);
    }
    MPI_Finalize();
    return 0;
}


%%writefile sinal_par.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#ifndef N
#define N       1200000
#endif
#ifndef TOL
#define TOL     0.01
#endif
#ifndef MAX_IT
#define MAX_IT  100000
#endif
#define B_ESQ   50.0
#define B_DIR  -50.0

int main(int argc, char *argv[]) {
    int rank, nprocs;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &nprocs);


    int n = N / nprocs;
    double *s = malloc((n + 4) * sizeof(double));
    double *novo = malloc((n + 4) * sizeof(double));
    double *dados = NULL;

    if (s == NULL || novo == NULL) {
        fprintf(stderr, "Rank %d: erro de alocacao de memoria.\n", rank);
        // Aborta todos os processos se um deles nao conseguiu alocar memoria.
        MPI_Abort(MPI_COMM_WORLD, 1);
        return 1;
    }

    if (rank == 0) {
        dados = malloc(N * sizeof(double));
        if (dados == NULL) {
            fprintf(stderr, "Erro de alocacao do sinal global.\n");
            MPI_Abort(MPI_COMM_WORLD, 1);
            return 1;
        }
        srand(7);
        for (int i = 0; i < N; i++) {
            dados[i] = rand() % 200 - 100;
        }
    }

    // Distribui N amostras geradas pelo rank 0 em blocos de n amostras.
    MPI_Scatter(dados, n, MPI_DOUBLE,
                &s[2], n, MPI_DOUBLE,
                0, MPI_COMM_WORLD);

    int esquerda;
    int direita;
    if (rank == 0) {
        esquerda = MPI_PROC_NULL;
    } else {
        esquerda = rank - 1;
    }
    if (rank == nprocs - 1) {
        direita = MPI_PROC_NULL;
    } else {
        direita = rank + 1;
    }

    // Todos devem estar na mesma linha antes de começar o while loop
    MPI_Barrier(MPI_COMM_WORLD);
    double t0 = MPI_Wtime();
    int it = 0;
    int mudaram_global;

    do {
        if (rank == 0) {
            s[0] = B_ESQ;
            s[1] = B_ESQ;
        }
        if (rank == nprocs - 1) {
            s[n + 2] = B_DIR;
            s[n + 3] = B_DIR;
        }

        // Envia as duas primeiras amostras para a esquerda; ao mesmo tempo,
        // recebe as duas primeiras amostras do vizinho da direita em seu halo direito.
        // MPI_PROC_NULL faz a operacao correspondente ser ignorada, sem travar.
        MPI_Sendrecv(&s[2], 2, MPI_DOUBLE, esquerda, 0,
                     &s[n + 2], 2, MPI_DOUBLE, direita, 0,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        // Envia as duas ultimas amostras para a direita; ao mesmo tempo,
        // recebe as duas ultimas amostras do vizinho da esquerda em seu halo esquerdo.
        // A troca casada de envio/recebimento evita deadlock.
        MPI_Sendrecv(&s[n], 2, MPI_DOUBLE, direita, 1,
                     &s[0], 2, MPI_DOUBLE, esquerda, 1,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        int mudaram_local = 0;
        for (int i = 2; i <= n + 1; i++) {
            novo[i] = (s[i - 2] + s[i - 1] + s[i] + s[i + 1] + s[i + 2]) / 5.0;
            if (fabs(novo[i] - s[i]) >= TOL) {
                mudaram_local++;
            }
        }

        double *tmp = s;
        s = novo;
        novo = tmp;

        // O divulga o resultado o resultado para TODOS.
        // Assim, nenhum processo para enquanto ainda ha mudancas em outro bloco.
        MPI_Allreduce(&mudaram_local, &mudaram_global, 1,
                      MPI_INT, MPI_SUM, MPI_COMM_WORLD);
        it++;
    } while (mudaram_global > 0 && it < MAX_IT);

    double tempo_local = MPI_Wtime() - t0;
    double tempo_max = 0.0;

    // Coleta o maior tempo entre processos: a execucao paralela so termina
    // quando o ultimo processo conclui seu trabalho.
    MPI_Reduce(&tempo_local, &tempo_max, 1,
               MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);

    // Reunir os dados de todos os processos em um único processo.
    // preservando a ordem original para somar como na versao sequencial.
    MPI_Gather(&s[2], n, MPI_DOUBLE,
               dados, n, MPI_DOUBLE,
               0, MPI_COMM_WORLD);

    if (rank == 0) {
        double soma = 0.0;
        for (int i = 0; i < N; i++) {
            soma += dados[i];
        }
        printf("Iteracoes = %d\n", it);
        printf("Soma      = %.6f\n", soma);

        printf("s[2] = %.6f   s[N/2] = %.6f   s[N+1] = %.6f\n",
               dados[0], dados[N / 2 - 2], dados[N - 1]);
        printf("Tempo = %.2f s\n", tempo_max);
    }

    free(dados);
    free(s);
    free(novo);

    MPI_Finalize();
    return 0;
}



!mpicc -O2 -Wall -o sinal_par sinal_par.c -lm
!for p in 1 2 3 4; do echo "== $p processo(s)"; mpirun --allow-run-as-root -n $p -host localhost:4 ./sinal_par; done

== 1 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 5.35 s
== 2 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 7.15 s
== 3 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 9.81 s
== 4 processo(s)
Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 10.00 s *** SDequerncial: !mpicc -O2 -Wall -o sinal_seq sinal_seq.c -lm && mpirun --allow-run-as-root -n 1 ./sinal_seq  Iteracoes = 1279
Soma      = -596965.773600
s[2] = 48.902079   s[N/2] = 5.697111   s[N+1] = -48.839417
Tempo = 5.04 s


```




- [7]
0s
!lscpu | grep -E "Model name|CPU\(s\)|Thread|Core|Socket"
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1